# `transform.ipynb` - Limpieza, normalización y validación

Funciones puras (sin efectos secundarios de logging/IO) que implementan las reglas de negocio de limpieza, normalización y validación de calidad de datos exigidas por la especificación del proyecto GeoStat.

Cada función incluye una celda de **prueba rápida** justo debajo, para poder ejecutarla de forma aislada y ver el resultado sin correr el pipeline completo.

> Depende de `config.ipynb` (debe haberse ejecutado antes con `%run`, ya que usa `FACTOR_SQMI_A_KM2`, `TASAS_CAMBIO_EUR`, `SINONIMOS_PAIS` y `PAISES_EUROPA_VALIDOS`).

In [ ]:
import re
import unicodedata

## 1. Normalización de identificadores de país

Requisito: *"Normaliza las cadenas de texto de los países para cruzarlas entre la API y SQLite sin problemas de mayúsculas, acentos o espacios en blanco residuales."*

`normalizar_nombre_pais` aplica, en orden:

1. `re.sub(r"\s+", " ", texto.strip())` -> colapsa espacios múltiples y recorta los de los extremos.
2. `quitar_acentos(...)` -> elimina tildes/diacríticos vía `unicodedata`.
3. `.upper()` -> mayúsculas.
4. Traducción por `SINONIMOS_PAIS` -> convierte nombres en español a su forma canónica en inglés.

In [ ]:
def quitar_acentos(texto: str) -> str:
    """Elimina tildes/diacríticos ('España' -> 'Espana')."""
    nfkd = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


def normalizar_nombre_pais(nombre_original: str) -> str:
    """
    Estandariza el nombre de un país para poder cruzarlo entre la API
    y la SQLite sin problemas de mayúsculas, acentos o espacios
    residuales, y traduce nombres en español a su forma canónica en
    inglés mediante SINONIMOS_PAIS.
    """
    if nombre_original is None:
        return ""
    limpio = re.sub(r"\s+", " ", nombre_original.strip())
    limpio = quitar_acentos(limpio).upper()
    limpio = SINONIMOS_PAIS.get(limpio, limpio)
    return limpio

In [ ]:
# Prueba rápida
for ejemplo in ["  Cyprus  ", "ESPAÑA", "francia", "  montenegro", "BULGARIA"]:
    print(f"{ejemplo!r:20s} -> {normalizar_nombre_pais(ejemplo)!r}")

## 2. Unificación de unidades de superficie

Requisito: *"Detecta las mediciones expresadas en millas cuadradas (sq_mi) y transfórmalas a kilómetros cuadrados."*

In [ ]:
def convertir_superficie_a_km2(valor: float, unidad: str) -> float:
    """Convierte sq_mi -> km2. Si ya viene en km2, lo deja igual."""
    if valor is None:
        return None
    unidad_norm = (unidad or "").strip().lower()
    if unidad_norm == "sq_mi":
        return round(valor * FACTOR_SQMI_A_KM2, 3)
    return round(valor, 3)

In [ ]:
# Prueba rápida: Luxemburgo son 998 sq_mi (~2585 km2 reales)
print(convertir_superficie_a_km2(998.0, "sq_mi"))
print(convertir_superficie_a_km2(56594.0, "km2"))

## 3. Unificación monetaria

Requisito: *"Convierte las diferentes divisas locales a Euros (EUR) aplicando una estructura de tipos de cambio."*

Si llega una divisa no contemplada en `TASAS_CAMBIO_EUR`, se lanza `ValueError` — en `extract.ipynb` esto se captura para desviar el registro a cuarentena en lugar de detener el pipeline.

In [ ]:
def convertir_pib_a_eur(valor: float, divisa: str) -> float:
    """
    Convierte un importe de PIB expresado en `divisa` a EUR usando la
    estructura de tipos de cambio TASAS_CAMBIO_EUR (1 EUR = X divisa).
    Lanza ValueError si la divisa no está soportada, para que el
    registro se pueda desviar a cuarentena en vez de romper el pipeline.
    """
    if valor is None:
        return None
    divisa_norm = (divisa or "").strip().upper()
    tasa = TASAS_CAMBIO_EUR.get(divisa_norm)
    if tasa is None:
        raise ValueError(f"Divisa no soportada en la estructura de cambio: {divisa_norm}")
    return valor / tasa

In [ ]:
# Prueba rápida
print(round(convertir_pib_a_eur(1000, "USD"), 2), "EUR")
print(round(convertir_pib_a_eur(1000, "EUR"), 2), "EUR")

# Prueba de la divisa no soportada (debe lanzar ValueError)
try:
    convertir_pib_a_eur(1000, "JPY")
except ValueError as error:
    print(f"ValueError esperado: {error}")

## 4. Calidad de datos - reglas de cuarentena

Requisito: *"Filtra los registros con valores numéricos incoherentes (PIB ≤ 0) o entidades territoriales no pertenecientes a la región europea, desviándolos a la tabla de cuarentena."*

In [ ]:
def es_pib_incoherente(pib_valor) -> bool:
    """Regla de negocio: PIB <= 0 se considera incoherente."""
    return pib_valor is None or pib_valor <= 0


def es_pais_europeo_valido(nombre_normalizado: str) -> bool:
    """
    True si, tras normalizar, el nombre corresponde a uno de los
    países europeos reconocidos (PAISES_EUROPA_VALIDOS). Filtra
    entidades ficticias / no europeas como 'Narnia' o 'Atlantis'.
    """
    return nombre_normalizado in PAISES_EUROPA_VALIDOS

In [ ]:
# Prueba rápida
print(es_pib_incoherente(-1.0), es_pib_incoherente(500.0))
print(es_pais_europeo_valido(normalizar_nombre_pais("Narnia")))
print(es_pais_europeo_valido(normalizar_nombre_pais("Francia")))